# Модуль 05 · Урок 1. Даты, интервалы и часовые пояса

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Строка даты — ещё не дата
Строка `'2026-04-05'` выглядит как дата, но не поддерживает корректную временную арифметику. `pd.to_datetime(...,errors='coerce')` преобразует в тип datetime, а некорректные строки — в `NaT`. Не полагайтесь на угадывание форматов вроде `03/04/2026`: для неоднозначных источников указывайте `format='%d/%m/%Y'` или нужный порядок явно. Разница дат вычисляется через вычитание и имеет тип Timedelta.

In [ ]:
import pandas as pd
subscriptions = pd.DataFrame({'start':['2026-02-01','2026-02-10'],'end':['2026-02-08','2026-03-01']})
start = pd.to_datetime(subscriptions.start)
end = pd.to_datetime(subscriptions.end)
print((end-start).dt.days)

## 2. Аксессор dt и периодизация
После преобразования Series в datetime можно использовать `.dt.year`, `.dt.month`, `.dt.day`, `.dt.day_name()`, `.dt.to_period('M')`. Период `2026-03` удобнее текста при группировке по месяцам: он имеет собственные правила сравнения и сортировки. При работе с датами храните реальные datetime, а отображение форматируйте только в самом конце.

In [ ]:
weeks = pd.Series(pd.to_datetime(['2026-04-01','2026-04-08','2026-05-02']))
print(weeks.dt.to_period('M'))
print(weeks.dt.dayofweek)  # понедельник=0

## 3. Время и часовые пояса
Наивная дата (`datetime64` без timezone) не указывает временную зону. События в международной системе лучше сохранять в UTC, а затем конвертировать в местное время. `tz_localize('UTC')` **назначает** зону наивному времени; `tz_convert('Europe/Amsterdam')` **переводит** уже осведомлённую дату в другую зону. Перепутанные операции могут незаметно сдвинуть отчёты на часы.

In [ ]:
utc = pd.to_datetime(pd.Series(['2026-07-10 12:00','2026-07-11 12:00'])).dt.tz_localize('UTC')
print(utc.dt.tz_convert('Europe/Amsterdam'))

## 4. Временные интервалы и ограничения
`pd.Timestamp`, `pd.Timedelta` и `pd.Period` решают разные задачи: точный момент, продолжительность, период календаря. Для фильтрации интервалов можно использовать сравнения datetime, но нужно явно решить, включать ли границы. Для месячных метрик обычно используется дата **события**, а не дата загрузки файла.

**Проверьте себя:** чем `tz_localize` отличается от `tz_convert`? Зачем указывать формат для неоднозначной даты? Что вернёт `.dt.to_period('M')`?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m5_01 · add month

Преобразуйте текстовую date в datetime и добавьте колонку `month` типа Period[M]. Исходную date оставьте строковой.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_01: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_01 = expected_for("m5_01", df=df)


In [ ]:
# m5_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 11 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`, `month`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment    month
z  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...  2026-01
b  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка  2026-01
a  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно  2026-01
d  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок  2026-02
x  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                   2026-02
q  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None  2026-02
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_01", result, _expected_m5_01)

### m5_02 · days between

Получите целое количество дней между строковыми датами `end` и `start`. Порядок важен: отрицательные значения возможны.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `start` — начальная дата, строка или Timestamp; в этом примере: `'2026-01-05'`.
- `end` — конечная дата, строка или Timestamp; в этом примере: `'2026-02-01'`.


**Выполнение:** используйте готовые переменные `start`, `end`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_02: используются переменные start, end

start = '2026-01-05'

end = '2026-02-01'


# Снимок правильного результата для автопроверки.
_expected_m5_02 = expected_for("m5_02", start=start, end=end)


In [ ]:
# m5_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: start, end
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`int`. Результат для показанного примера: `27`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_02", result, _expected_m5_02)

### m5_03 · weekday index

Получите Series с номером дня недели каждого заказа (`пн=0`), сохранив индекс исходного DataFrame.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_03: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_03 = expected_for("m5_03", df=df)


In [ ]:
# m5_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 6 значений; имя серии: `date`.

```text
z    5
b    6
a    6
d    6
x    0
q    3
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_03", result, _expected_m5_03)

### m5_04 · filter dates

Выберите заказы, date которых попадает в замкнутый интервал [start,finish]. Даты приходят строками ISO.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `start` — начальная дата, строка или Timestamp; в этом примере: `'2026-01-04'`.
- `finish` — конечная дата, строка или Timestamp; в этом примере: `'2026-02-01'`.


**Выполнение:** используйте готовые переменные `df`, `start`, `finish`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_04: используются переменные df, start, finish

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

start = '2026-01-04'

finish = '2026-02-01'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m5_04 = expected_for("m5_04", df=df, start=start, finish=finish)


In [ ]:
# m5_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, start, finish
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount status        date     comment
b  demo_o2          2   Тверь     кофе  247.0    1        10    new  2026-01-04  без звонка
a  demo_o3          1  Москва      чай   97.0    4         5   paid  2026-01-04      Срочно
d  demo_o4          3  Казань    книги  317.0    1         0   paid  2026-02-01     подарок
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_04", result, _expected_m5_04)

### m5_05 · local hours

На входе Series UTC-строк без зоны. Получите Series с местным часом в зоне Europe/Amsterdam после явной локализации к UTC.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `times` — `pd.Series` (индекс и значения определены в ячейке примера).


**Выполнение:** используйте готовые переменные `times`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m5_05: используются переменные times

times = pd.Series(['2026-01-01 12:00', '2026-07-01 12:00'], index=[0, 1])

print("times — демонстрационные данные:")

display(times)


# Снимок правильного результата для автопроверки.
_expected_m5_05 = expected_for("m5_05", times=times)


In [ ]:
# m5_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: times
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `None`.

```text
0    13
1    14
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m5_05", result, _expected_m5_05)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()